![](automated ticket pipeline.jpg)

####1. Automated Support Triage & Ticket Routing
Customer service teams are often overwhelmed with a massive backlog of tickets. Instead of having humans read every ticket to decide who handles it, your pipeline does it instantly.

####NLP Techniques (Syntax & Structure)
- Tokenization: Splits text into individual words and punctuation.
- POS Tagging: Assigns grammatical labels (nouns, verbs, adjectives).

####NLU Techniques (Meaning & Context)
- Named Entity Recognition (NER): Extracts specific real-world subjects (products, locations).
- Sentiment Analysis: Identifies emotional tone (Positive, Negative, Neutral).
- Intent Detection: Determines the user's specific goal (returns, complaints).
- Vector Embeddings: Converts text meaning into mathematical numbers.

####How to create Databricks Token
- In your Databricks workspace, click your username in the top bar and select Settings.
- Click Developer.
- Next to Access tokens, click Manage.
- Click Generate new token.

In [0]:
#databricks secrets create-scope izgenaiscope
#databricks secrets put-secret izgenaiscope databricks_token
DATABRICKS_TOKEN = dbutils.secrets.get(scope="izgenaiscope", key="databricks_token")

1. We are getting data into the Bronze layer of our Medallion Arch.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakehousecat_ai;
CREATE SCHEMA IF NOT EXISTS lakehousecat_ai.lakehouseschema_ai;
drop table if exists lakehousecat_ai.lakehouseschema_ai.customer_reviews;
CREATE TABLE IF NOT EXISTS lakehousecat_ai.lakehouseschema_ai.customer_reviews (
  review_id STRING,
  city string,
  review_text STRING
);
INSERT INTO lakehousecat_ai.lakehouseschema_ai.customer_reviews VALUES
("REV-001",'NYC', "The shoe are beautiful, but they ripped after two days of wearing them!. I want my money back"),
("REV-002",'NY', "The Laptop Shipping took only 3 weeks. Absolutely unacceptable."),
("REV-003",'CA', "Perfect fit! Will definitely be buying from you guys again."),
("REV-004",'CAL', "I received the wrong color. I ordered black but got blue. How do I fix this?"),
("REV-005",'CHE', "The product was bad, I don't to reorder the same product again");;


In [0]:
df_reviews = spark.read.table("lakehousecat_ai.lakehouseschema_ai.customer_reviews")
display(df_reviews)

####NLP & NLU using Text Generation Model

In [0]:
%sql
CREATE OR REPLACE VIEW lakehousecat_ai.lakehouseschema_ai.advanced_text_analysis AS
SELECT 
  review_id,
  review_text,

  -- NLP CONCEPTS: Syntax, Structure, & Mechanics

  -- 1. Tokenization: Breaking text into individual pieces
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('You are a tokenizer. Break the following text down into individual words and punctuation marks. Return ONLY a comma-separated list of tokens: ', review_text)
  ) AS nlp_tokens,

  -- 2. POS (Part-of-Speech) Tagging: Identifying grammatical labels
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('You are a POS tagger. Identify the part of speech for the words in this text. Return ONLY a comma-separated list in "word(TAG)" format (e.g., fast(ADJECTIVE), run(VERB)): ', review_text)
  ) AS nlp_pos_tags,

  -- NLU CONCEPTS: Meaning, Intent, Sentiment & Context

  -- 3. NER (Named Entity Recognition): Extracting specific real-world subjects
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('You are an NER tool., Extract Named Entities such as Person, Organization, Duration, Time, Money, Location, or Product. Return ONLY a comma-separated list in "Entity (Type)" format. If none exist, return "None": ', review_text)
  ) AS nlu_ner_entities,

  -- 4. Sentiment Analysis: Understanding emotional tone
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('You are a sentiment analyzer. Classify the emotional tone of this review. Return ONLY the word POSITIVE, NEGATIVE, or NEUTRAL: ', review_text)
  ) AS nlu_sentiment,

  -- 5. Semantic Matching (Similarity Scoring): Checking for a specific underlying goal (This example is to show the similarity scoring used behind)
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('Does this review express an intent to return the product, ask for a refund, or complain about shipping? Return ONLY the word YES or NO: ', review_text)
  ) AS nlu_return_semantic_matching,

  -- 6. Intent Classification/Generation: Identifying the broad purpose of the user
  ai_query(
    'databricks-meta-llama-3-1-405b-instruct',
    CONCAT('Identify the primary intent of this customer review. Choose ONLY ONE from the following categories: "Product Inquiry", "Complaint", "Praise", "Feature Request", "Customer Support", or "Other". Return ONLY the category name: ', review_text)
  ) AS nlu_primary_intent,

  -- 7. Vector Embeddings: Converting meaning into mathematical representations
  -- Note: Text generation models (like Llama) don't create embeddings. 
  ai_query(
    'databricks-bge-large-en', 
    review_text
  ) AS nlu_vector_values

FROM lakehousecat_ai.lakehouseschema_ai.customer_reviews;

SELECT * FROM lakehousecat_ai.lakehouseschema_ai.advanced_text_analysis;

![](automated mail generator.jpg)

####2. Automated Customer Review Response Generator (E-Commerce)
Instead of just classifying a negative review, We can use LLM (NLG) to actually write the email response to the customer.

The Pipeline Concept: Read the negative review from a Delta table, pass it to the LLM with strict instructions, and output a drafted email into a new column.

LLM & GenAI: Using Llama understand the nuanced frustration in the review.

NLG (Natural Language Generation): The model synthesizes a polite, grammatically perfect, and empathetic email.

Grounding: Pass the company's official return policy into the system prompt. The AI is grounded in this specific document so it doesn't make up rules.

Guardrailing: Add a strict rule to the prompt: "Never promise a refund or discount. Only offer to connect them to the support team."

Hallucination (Anti) Mitigation: Set temperature=0.1, so the AI doesn't invent fake product features or invent a fake customer service rep name.

In [0]:
%sql
CREATE OR REPLACE TABLE lakehousecat_ai.lakehouseschema_ai.ecommerce_raw_reviews (
  review_id STRING,
  customer_review STRING
);

INSERT INTO lakehousecat_ai.lakehouseschema_ai.ecommerce_raw_reviews (review_id, customer_review)
VALUES 
  ('REV-001', 'The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.'),
  ('REV-002', 'Shipping took 3 weeks. Absolutely unacceptable.'),
  ('REV-003', 'Perfect fit! Will definitely be buying from you guys again.'),
  ('REV-004', 'I received the wrong color. I ordered black but got blue. How do I fix this?'),
  ('REV-004', 'I received the product since purchased 45 days?, i want to return the product');

SELECT * FROM lakehousecat_ai.lakehouseschema_ai.ecommerce_raw_reviews;

In [0]:
%sql
-- Assuming your raw data is still in 'default.ecommerce_raw_reviews'

SELECT 
    review_id,
    customer_review,
    ai_query(
        -- 1. The Model Endpoint
        'databricks-meta-llama-3-1-405b-instruct',
        
        -- 2. The Prompt (System Instructions + The Data Column)
        CONCAT(
            'You are an empathetic, professional customer support agent for an E-Commerce shoe company. ',
            'Read the customer review and write a direct email response to them. ',
            
            'GROUNDING CONTEXT: ',
            '- We only accept returns within 30 days of purchase. ',
            '- We do NOT give cash refunds. We only offer store credit or exact item replacements. ',
            
            'GUARDRAILS: ',
            '1. Never promise a refund through electronic money transfer under any circumstances. ',
            '2. Never offer a discount code or coupon. ',
            '3. Keep the response under 4 sentences. ',
            
            'ANTI HALLUCINATION: ',
            'If the user asks for something not covered in the policies above, do not invent a solution. ',
            'Simply state: "I am escalating this to our senior support team who will contact you within 24 hours." ',
            
            'Customer Review: ', customer_review
        ),
        
        -- 3. The Model Parameters (Ensuring robotic/strict adherence to guardrails)
        modelParameters => named_struct('temperature', 0.1, 'max_tokens', 1500)
        
    ) AS ai_generated_email

FROM lakehousecat_ai.lakehouseschema_ai.ecommerce_raw_reviews;